## Importation des bibliothèques et des données

In [43]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import io
import requests
import numpy as np
import json
from shapely.geometry import shape
np.float = float
np.long = int
import sys
sys.modules["numba"] = None



## 2. Import des données depuis les URLs

Nos 3 sources sont :
- **`df_ips`** : l'indice de position sociale (IPS) de chaque lycée, par année (data.education.gouv.fr) ;
- **`df_res`** : les indicateurs de résultat des lycées généraux et technologiques (taux de réussite, de mentions, d'accès…) (data.education.gouv.fr) ;
- **`df_carte`** : les contours géographiques des départements (ADEME), nécessaires pour les cartes.

La fonction `recup_data` télécharge un CSV 
Pour `df_carte`, la colonne `geometry` arrive sous forme de texte JSON : on la convertit en objets géométriques pour obtenir un `GeoDataFrame`.

In [44]:
url_ips = "https://data.education.gouv.fr/explore/dataset/fr-en-ips_lycees/download/?format=csv&timezone=Europe/Berlin&lang=fr&use_labels_for_header=true&csv_separator=%3B"
url_res = "https://data.education.gouv.fr/explore/dataset/fr-en-indicateurs-de-resultat-des-lycees-gt_v2/download/?format=csv&timezone=Europe/Berlin&lang=fr&use_labels_for_header=true&csv_separator=%3B"
url_carte = "https://data-interne.ademe.fr/data-fair/api/v1/datasets/geo-contours-departements/lines?draft=false&size=10000&finalizedAt=2022-03-02T14:11:30.661Z&page=1&format=csv&sep=;"

df_ips = pd.read_csv(io.StringIO(requests.get(url_ips).text), sep=";")
df_res = pd.read_csv(io.StringIO(requests.get(url_res).text), sep=";")
df_carte = pd.read_csv(url_carte, sep=";") 

df_ips.head(5)

,Rentrée scolaire,Académie,Code du département,Département,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur,Type de lycée,IPS voie GT,IPS voie PRO,IPS Ensemble GT-PRO,Ecart-type de l'IPS voie GT,Ecart-type de l'IPS voie PRO
0,2019-2020,CLERMONT-FERRAND,3.0,ALLIER,0030044G,LYCEE GENERAL BLAISE DE VIGENERE,3254.0,SAINT POURCAIN SUR SIOULE,public,LEGT,109.9,NaN,109.9,34.7,NaN
1,2019-2020,CLERMONT-FERRAND,3.0,ALLIER,0030051P,LYCEE GENERAL ET TECHNOLOGIQUE ALBERT LONDRES ...,3095.0,CUSSET,public,LEGT,109.4,NaN,109.4,35.9,NaN
2,2019-2020,CLERMONT-FERRAND,3.0,ALLIER,0030084A,LYCEE GENERAL PRIVE SAINT BENOIT,3190.0,MOULINS,privé sous contrat,LEGT,105.6,NaN,105.6,37.8,NaN
3,2019-2020,CLERMONT-FERRAND,3.0,ALLIER,0030105Y,LYCEE PROFESSIONNEL PRIVE SAINTE LOUISE,3185.0,MONTLUCON,privé sous contrat,LP,NaN,81.1,81.1,NaN,31.6
4,2019-2020,CLERMONT-FERRAND,3.0,ALLIER,0030905T,LYCEE PROFESSIONNEL GUSTAVE EIFFEL LYCEE DES M...,3118.0,GANNAT,public,LP,NaN,89.2,89.2,NaN,27.5


In [45]:
df_res.head(5)

,Année,UAI,Etablissement,Secteur,Code commune,Commune,Code departement,Département,Académie,Code région,...,Valeur ajoutée du taux de mentions - S2TMD,Valeur ajoutée du taux de mentions - STHR,Nombre de mentions TB avec félicitations - G,Nombre de mentions TB sans félicitations - G,Nombre de mentions B - G,Nombre de mentions AB - G,Nombre de mentions TB avec félicitations - T,Nombre de mentions TB sans félicitations - T,Nombre de mentions B - T,Nombre de mentions AB - T
0,2017,0290108K,LYCEE DUPUY DE LOME (GENERAL ET TECHNO.),public,29019,BREST,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017,0290156M,LYCEE ST LOUIS,privé sous contrat,29026,CHATEAULIN,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017,0290174G,LYCEE NOTRE DAME DE KERBERTRA,privé sous contrat,29233,QUIMPERLE,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017,0290177K,LYCEE DU KREISKER,privé sous contrat,29259,ST POL DE LEON,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017,0290181P,LYCEE JAVOUHEY,privé sous contrat,29019,BREST,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ici on a importé les deux tableaux dont on a besoin avec l'IPS (indice de position sociale) et les résultats à l'examen du baccalauréat. On ne les a pas importé directement avec pd.read_csv car cela bloquait parfois selon l'envirronement python du terminal utilisé. Avec la bibliothèque requests adaptée aux API au moins on est sur de ne pas avoir ce problème de versions.
De plus on a importé un shapefile avec les délimitations administratives des départements français pour pouvoir faire des cartes plus tard.

## Nettoyage et mise en forme des données

On cherche a voir quelles variables sont pertinentes pour répondre a notre problématique, pour ce faire, on regarde le nombre de N.A. et le pourcentage de N.A. par colonne qui nous seront potentiellement utile dans l'analyse des données.

In [ ]:
cols_ips = [
    "UAI", "Rentrée scolaire", "Nom de l'établissment", "Type de lycée", "Secteur",
    "Académie", "Département", "Code du département", "Nom de la commune", "Code INSEE de la commune",
    "IPS voie GT", "Ecart-type de l'IPS voie GT", "IPS Ensemble GT-PRO",
]

cols_res = [
    "UAI", "Année", "Code région", "Code departement", "Département", "Secteur", "Code commune", "Commune",
    "Taux de réussite - Toutes séries", "Taux de mentions - Toutes séries",
    "Valeur ajoutée du taux de réussite - Toutes séries",
    "Taux d'accès 2nde-bac", "Taux d'accès 1ere-bac", "Taux d'accès terminale-bac",
    "Présents - Gnle", "Nombre de mentions TB sans félicitations - G",
    "Taux de réussite - Gnle", "Taux de mentions - Gnle",
    "Taux de réussite - STMG", "Taux de mentions - STMG",
]

cols_carte = ["DDEP_C_COD", "DDEP_L_LIB", "DREG_L_LIB", "geometry"]

for nom, df, cols in [("df_ips", df_ips, cols_ips),
                      ("df_res", df_res, cols_res),
                      ("df_carte", df_carte, cols_carte)]:
    cols_ok = [c for c in cols if c in df.columns]
    resume = pd.DataFrame({
        "NA": df[cols_ok].isna().sum(),
        "% NA": (df[cols_ok].isna().mean() * 100).round(1)
    }).sort_values("NA", ascending=False)
    print(f"--- {nom} ({len(df)} lignes) ---")
    print(resume.to_string())
    print()

--- df_ips (21777 lignes) ---
                                NA  % NA
Ecart-type de l'IPS voie GT  14366  66.0
IPS voie GT                   6985  32.1
Code du département             90   0.4
Code INSEE de la commune        90   0.4
UAI                              0   0.0
Secteur                          0   0.0
Type de lycée                    0   0.0
Nom de l'établissment            0   0.0
Rentrée scolaire                 0   0.0
Nom de la commune                0   0.0
Département                      0   0.0
Académie                         0   0.0
IPS Ensemble GT-PRO              0   0.0

--- df_res (32485 lignes) ---
                                                       NA  % NA
Nombre de mentions TB sans félicitations - G        23892  73.5
Taux de mentions - Gnle                             21558  66.4
Taux de réussite - Gnle                             21558  66.4
Présents - Gnle                                     21558  66.4
Taux de mentions - STMG                      

On voit ici que certaines colonnes ont beaucoup de N.A. mais la plupart sont du a des lycées n'etant que technologiques ou que générales. On enlevera donc simplement les lignes avec ces N.A. quand necessaire pour les analyses mais pas la peine de supprimer les lignes pour tout le projet

In [47]:
df_ips[df_ips["Code du département"].isna()].head(5)

,Rentrée scolaire,Académie,Code du département,Département,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur,Type de lycée,IPS voie GT,IPS voie PRO,IPS Ensemble GT-PRO,Ecart-type de l'IPS voie GT,Ecart-type de l'IPS voie PRO
602,2019-2020,CORSE,NaN,CORSE-DU-SUD,6200004K,LYCEE PROFESSIONNEL FINOSELLO AJACCIO,NaN,AJACCIO,public,LP,NaN,86.2,86.2,NaN,27.3
603,2019-2020,CORSE,NaN,HAUTE-CORSE,7200021K,LYCEE GENERAL ET TECHNOLOGIQUE PASCAL PAOLI CORTE,NaN,CORTE,public,LEGT,111.8,NaN,111.8,31.4,NaN
1845,2016-2017,CORSE,NaN,CORSE-DU-SUD,6200001G,LYCEE GENERAL ET TECHNOLOGIQUE FESCH AJACCIO,NaN,AJACCIO,public,LEGT,113.2,NaN,113.2,NaN,NaN
1846,2016-2017,CORSE,NaN,CORSE-DU-SUD,6200002H,LYCEE GENERAL ET TECHNOLOGIQUE LAETITIA BONAPA...,NaN,AJACCIO,public,LEGT,112.5,NaN,112.5,NaN,NaN
1847,2016-2017,CORSE,NaN,CORSE-DU-SUD,6200004K,LYCEE PROFESSIONNEL FINOSELLO AJACCIO,NaN,AJACCIO,public,LP,NaN,87.6,87.6,NaN,NaN


Ici on voit que les résultats au bac des lycées corses sont manquants donc on va enlever ces lignes dans le tableau sur l'IPS. 
Ensuite on va créer une colonne correspondant à l'année dans df_ips pour pouvoir merge les deux tableaux ensemble sur le lycée et l'année scolaire.

In [48]:
df_ips = df_ips[df_ips["Code du département"].notna()]
df_ips["Année"] = df_ips["Rentrée scolaire"].str[-4:].astype(int)
df_ips["Code du département"] = pd.to_numeric(df_ips["Code du département"], errors="coerce").astype("Int64")
df_res["Code departement"] = pd.to_numeric(df_res["Code departement"], errors="coerce").astype("Int64")
df.head(5)

,geometry,DDEP_C_COD,DDEP_L_LIB,DREG_L_LIB
0,"{""type"":""Polygon"",""coordinates"":[[[6.1684513,4...",01,Ain,Auvergne-Rhône-Alpes
1,"{""type"":""Polygon"",""coordinates"":[[[4.2557339,4...",02,Aisne,Hauts-de-France
2,"{""type"":""Polygon"",""coordinates"":[[[4.0045583,4...",03,Allier,Auvergne-Rhône-Alpes
3,"{""type"":""Polygon"",""coordinates"":[[[6.9670858,4...",04,Alpes-de-Haute-Provence,Provence-Alpes-Côte d'Azur
4,"{""type"":""Polygon"",""coordinates"":[[[7.0758717,4...",05,Hautes-Alpes,Provence-Alpes-Côte d'Azur


On peut désormais joindre sur l'année scolaire (2022 pas 2021-2022) et sur le lycée

In [49]:
# Jointure sur UAI + Année
df = df_ips.merge(df_res, on=["UAI","Année"], how="left", suffixes=("_ips","_res"))
df.head(5) 

,Rentrée scolaire,Académie_ips,Code du département,Département_ips,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur_ips,Type de lycée,...,Valeur ajoutée du taux de mentions - S2TMD,Valeur ajoutée du taux de mentions - STHR,Nombre de mentions TB avec félicitations - G,Nombre de mentions TB sans félicitations - G,Nombre de mentions B - G,Nombre de mentions AB - G,Nombre de mentions TB avec félicitations - T,Nombre de mentions TB sans félicitations - T,Nombre de mentions B - T,Nombre de mentions AB - T
0,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030044G,LYCEE GENERAL BLAISE DE VIGENERE,3254.0,SAINT POURCAIN SUR SIOULE,public,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030051P,LYCEE GENERAL ET TECHNOLOGIQUE ALBERT LONDRES ...,3095.0,CUSSET,public,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030084A,LYCEE GENERAL PRIVE SAINT BENOIT,3190.0,MOULINS,privé sous contrat,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030105Y,LYCEE PROFESSIONNEL PRIVE SAINTE LOUISE,3185.0,MONTLUCON,privé sous contrat,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030905T,LYCEE PROFESSIONNEL GUSTAVE EIFFEL LYCEE DES M...,3118.0,GANNAT,public,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Pour la suite du projet, on crée de nouvelles variables plus facile a interpréter : 
   - `ecart_selectivite` : taux de réussite moins taux d'accès de la seconde au bac. Plus il est grand, plus le lycée « perd » d'élèves en route ;
   - `taux_TB` : part des mentions Très Bien parmi les présents en voie générale ;
   - `quartile_IPS` : classement des lycées en 4 groupes selon leur IPS (Q1 = les moins favorisés, Q4 = les plus favorisés) ;
   - `taux_mentions_seconde_approx` : approximation du taux de mentions rapporté aux élèves entrés en seconde. 

In [ ]:
 # Variables créées
df["ecart_selectivite"] = df["Taux de réussite - Toutes séries"] - df["Taux d'accès 2nde-bac"]
df["taux_TB"] = df["Nombre de mentions TB sans félicitations - G"] / df["Présents - Gnle"] * 100
df["quartile_IPS"] = pd.qcut(df["IPS voie GT"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
df["taux_mentions_seconde_approx"] = df["Taux de mentions - Toutes séries"] * df["Taux d'accès 2nde-bac"] / 100
df.head(5)

,Rentrée scolaire,Académie_ips,Code du département,Département_ips,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur_ips,Type de lycée,...,Nombre de mentions B - G,Nombre de mentions AB - G,Nombre de mentions TB avec félicitations - T,Nombre de mentions TB sans félicitations - T,Nombre de mentions B - T,Nombre de mentions AB - T,ecart_selectivite,taux_TB,quartile_IPS,taux_mentions_seconde_approx
0,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030044G,LYCEE GENERAL BLAISE DE VIGENERE,3254.0,SAINT POURCAIN SUR SIOULE,public,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,Q2,59.86
1,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030051P,LYCEE GENERAL ET TECHNOLOGIQUE ALBERT LONDRES ...,3095.0,CUSSET,public,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,Q2,46.64
2,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030084A,LYCEE GENERAL PRIVE SAINT BENOIT,3190.0,MOULINS,privé sous contrat,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,Q2,30.42
3,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030105Y,LYCEE PROFESSIONNEL PRIVE SAINTE LOUISE,3185.0,MONTLUCON,privé sous contrat,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-2020,CLERMONT-FERRAND,3,ALLIER,0030905T,LYCEE PROFESSIONNEL GUSTAVE EIFFEL LYCEE DES M...,3118.0,GANNAT,public,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Enfin on met juste en forme le tableau avec les données géographiques.

In [52]:
df_carte["geometry"] = df_carte["geometry"].apply(
    lambda x: shape(json.loads(x)) if isinstance(x, str) else x)
df_carte["DDEP_C_COD"] = pd.to_numeric(df_carte["DDEP_C_COD"], errors="coerce").astype("Int64")
df_carte = df_carte.rename(columns={"DDEP_L_LIB":"Département","DDEP_C_COD":"Code du département", "DREG_L_LIB":"Région"})
df_carte = gpd.GeoDataFrame(df_carte, geometry="geometry")